In [1]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from autocorrect import Speller

df = pd.read_csv("UNITENReview.csv")

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
spell = Speller(lang='en')

contractions_dict = {
    "i'm": "i am", "it's": "it is", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "can't": "cannot", "won't": "will not", "isn't": "is not",
    "aren't": "are not", "we're": "we are", "they're": "they are", "you're": "you are"
}

slang_dict = {
    "w": "win", "yg": "yang", "tpi": "tapi", "je": "sahaja", "ni": "ini"
}


def preprocess_text(text):
    if pd.isna(text) or text == '#NAME?':
        return ""
    
    text = str(text)
    text = text.lower()
    for contraction, expansion in contractions_dict.items():
        text = text.replace(contraction, expansion)

    text = re.sub(r'[^\w\s]', ' ', text) 
    text = re.sub(r'\d+', '', text)      
    
    words = text.split()
    
    cleaned_words = []
    for word in words:
        if word in slang_dict:
            cleaned_words.append(slang_dict[word])
        else:
            cleaned_words.append(spell(word))
            
    words_no_stops = [word for word in cleaned_words if word not in stop_words]
    
    def get_wordnet_pos(word):
        tag = pos_tag([word])[0][1][0].upper()
        tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
        return tag_dict.get(tag, wordnet.NOUN)

    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words_no_stops]
    
    return " ".join(lemmatized)

print("Cleaning data... this might take a minute...")
df["Cleaned_Review"] = df["Review"].apply(preprocess_text)

df = df[df["Cleaned_Review"] != ""]

pd.set_option('display.max_colwidth', None)
print("\n--- SAMPLE RESULTS ---")
print(df[["Review", "Cleaned_Review"]].head(3))

Cleaning data... this might take a minute...

--- SAMPLE RESULTS ---
                                                                   Review  \
0                    Im happy with uniten actually, even the people are W   
1  I’m having a pretty good time here, happy to meet all of the W people.   
2                             a very neutral place in terms of everything   

                             Cleaned_Review  
0  im happy united actually even people win  
1    pretty good time happy meet win people  
2             neutral place term everything  


In [2]:
df.to_csv("Cleaned_UNITENReview.csv", index=False)

print("File successfully saved as Cleaned_UNITENReview.csv!")

File successfully saved as Cleaned_UNITENReview.csv!


In [ ]:
import pandas as pd
import re

# Load the dataset
df = pd.read_csv("UNITENReview.csv")

# Ensure the column is treated as text to avoid errors
reviews = df['Review'].astype(str)

# Create a list to hold our report lines
report = []
report.append("--- IDENTIFIED ISSUES IN 'Review' COLUMN ---\n")

# 1. Check for Missing/Null values or Excel errors (like #NAME?)
excel_errors = df[df['Review'] == '#NAME?']
null_values = df[df['Review'].isnull()]
report.append(f"1. Corrupted Data: Found {len(excel_errors)} Excel '#NAME?' error(s) and {len(null_values)} Null value(s).")

# 2. Check for Punctuation & Special Characters
# regex [^\w\s] looks for anything that is NOT a word character or whitespace
punct_count = reviews.apply(lambda x: bool(re.search(r'[^\w\s]', x))).sum()
report.append(f"2. Punctuation: {punct_count} reviews contain punctuation or special characters that need removal.")

# 3. Check for Numbers
# regex \d looks for digits
num_count = reviews.apply(lambda x: bool(re.search(r'\d', x))).sum()
report.append(f"3. Numbers: {num_count} reviews contain numbers/digits.")

# 4. Check for Mixed Casing
upper_count = reviews.apply(lambda x: any(c.isupper() for c in x)).sum()
report.append(f"4. Inconsistent Casing: {upper_count} reviews contain uppercase letters. Text needs to be converted to lowercase.")

# 5. Check for Contractions (e.g., don't, I'm)
contractions = ["n't", "'m", "'re", "'s", "'ll", "'ve", "'d"]
contraction_count = reviews.apply(lambda x: any(c in x.lower() for c in contractions)).sum()
report.append(f"5. Contractions: {contraction_count} reviews contain contractions that need expanding.")

# 6. Check for Slang (Internet and local Malaysian slang)
slangs = [' w ', ' yg ', ' tpi ', ' je ', ' ni ', ' tbh ', ' idk ']
# Add spaces around the word so we don't accidentally match "wife" for "w"
slang_count = reviews.apply(lambda x: any(slang in f" {x.lower()} " for slang in slangs)).sum()
report.append(f"6. Slang/Abbreviations: {slang_count} reviews contain common internet or local slang (e.g., 'yg', 'tpi', 'W').")

final_report_text = "\n".join(report)

print(final_report_text)

with open("Q1_Identified_Issues.txt", "w") as file:
    file.write(final_report_text)

print("\n✅ Success! The report has been saved as 'Q1_Identified_Issues.txt' in your folder.")